In [1]:
# Import OpenCV library for image processing
import cv2

# Import NumPy for numerical operations and matrix manipulation
import numpy as np

# =========================
# STEP 1: LOAD THE IMAGE
# =========================

# Read an image from the dataset folder.
# Replace "image1.jpg" with your own image name if needed.
image = cv2.imread(r"C:\Users\ahmad\Desktop\munich-germany-september-13-2024-260nw-2518377155.webp")

# Check whether the image was loaded successfully.
# If the image path is incorrect or the file doesn't exist,
# cv2.imread() returns None.
if image is None:
    print("Error: Image not found!")
    exit()

# =========================
# STEP 2: RESIZE THE IMAGE
# =========================

# Resize the image to a fixed size (800 x 600 pixels).
# This ensures that every image has the same dimensions,
# making processing faster and display more consistent.
image = cv2.resize(image, (800, 600))

# =========================
# STEP 3: CONVERT TO GRAYSCALE
# =========================

# Convert the color image (BGR) into a grayscale image.
# Grayscale images have only one intensity channel instead of three.
# This simplifies processing and reduces computation.
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# =========================
# STEP 4: APPLY GAUSSIAN BLUR
# =========================

# Apply Gaussian Blur to reduce image noise and smooth edges.
#
# (5,5) = Size of the Gaussian kernel.
# Larger kernels produce stronger smoothing.
#
# 0 = OpenCV automatically calculates the standard deviation.
blur = cv2.GaussianBlur(gray, (5, 5), 0)

# =========================
# STEP 5: ADAPTIVE THRESHOLDING
# =========================

# Convert the blurred grayscale image into a binary image.
#
# Pixels become either:
# 255 (White)
# 0   (Black)
#
# Adaptive threshold calculates the threshold separately
# for different regions of the image, making it useful
# for varying lighting conditions.
threshold = cv2.adaptiveThreshold(

    # Input grayscale image
    blur,

    # Maximum pixel value after thresholding
    255,

    # Use Gaussian weighted neighborhood
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,

    # Invert colors:
    # Objects become white
    # Background becomes black
    cv2.THRESH_BINARY_INV,

    # Size of neighborhood used for threshold calculation
    11,

    # Constant subtracted from the computed threshold
    2
)

# =========================
# STEP 6: MORPHOLOGICAL OPERATION
# =========================

# Create a 3x3 matrix filled with ones.
# This matrix is called the kernel.
kernel = np.ones((3, 3), np.uint8)

# Apply Morphological Closing.
#
# Closing = Dilation followed by Erosion.
#
# Purpose:
# - Fill small holes
# - Connect nearby white regions
# - Remove tiny black gaps
morph = cv2.morphologyEx(
    threshold,
    cv2.MORPH_CLOSE,
    kernel
)

# =========================
# STEP 7: FIND CONTOURS
# =========================

# Detect object boundaries (contours) in the binary image.
#
# RETR_EXTERNAL:
# Detect only the outermost contours.
#
# CHAIN_APPROX_SIMPLE:
# Compress contour points to save memory.
contours, _ = cv2.findContours(
    morph,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

# =========================
# STEP 8: DRAW CONTOURS
# =========================

# Make a copy of the original image.
# This preserves the original image while allowing us
# to draw detected contours on the copy.
output = image.copy()

# Draw all detected contours.
#
# -1 means draw every contour.
#
# (0,255,0) = Green color (BGR format)
#
# 2 = Line thickness
cv2.drawContours(
    output,
    contours,
    -1,
    (0, 255, 0),
    2
)

# =========================
# STEP 9: DISPLAY CONTOUR COUNT
# =========================

# Overlay the number of detected contours onto the image.
#
# len(contours) gives the total number of detected objects.
cv2.putText(

    # Image on which text is drawn
    output,

    # Text content
    f"Contours: {len(contours)}",

    # Position (x, y)
    (20, 40),

    # Font style
    cv2.FONT_HERSHEY_SIMPLEX,

    # Font size
    1,

    # Text color (Red)
    (0, 0, 255),

    # Thickness
    2
)

# =========================
# STEP 10: DISPLAY RESULTS
# =========================

# Show original image
cv2.imshow("Original Image", image)

# Show grayscale image
cv2.imshow("Grayscale", gray)

# Show blurred image
cv2.imshow("Gaussian Blur", blur)

# Show adaptive threshold result
cv2.imshow("Adaptive Threshold", threshold)

# Show morphological output
cv2.imshow("Morphological Output", morph)

# Show final image with contours
cv2.imshow("Detected Contours", output)

# Wait until a keyboard key is pressed
cv2.waitKey(0)

# Close all OpenCV windows
cv2.destroyAllWindows()

In [2]:
# ==========================================================
# PART 3 - YOLOv8 Object Detection
# ==========================================================

# Import OpenCV
import cv2

# Import YOLO model
from ultralytics import YOLO

# Load the YOLOv8 Nano model
# If yolov8n.pt is in the models folder, use:
# model = YOLO("models/yolov8n.pt")

model = YOLO("yolov8n.pt")

# Load the image
image = cv2.imread(r"C:\Users\ahmad\Desktop\munich-germany-september-13-2024-260nw-2518377155.webp")

# Check if the image exists
if image is None:
    print("Image not found!")
    exit()

# Run object detection
results = model(image)

# Get the annotated image
annotated_image = results[0].plot()

# Count detected objects
total_objects = len(results[0].boxes)

# Display object count
cv2.putText(
    annotated_image,
    f"Objects Detected: {total_objects}",
    (20, 40),
    cv2.FONT_HERSHEY_SIMPLEX,
    1,
    (0, 0, 255),
    2
)

# Save output image
cv2.imwrite("output/detected_image.jpg", annotated_image)

# Display result
cv2.imshow("YOLOv8 Object Detection", annotated_image)

cv2.waitKey(0)
cv2.destroyAllWindows()

print(f"Total Objects Detected: {total_objects}")
print("Output image saved in the output folder.")


0: 480x640 1 person, 1 bicycle, 3 cars, 665.6ms
Speed: 45.6ms preprocess, 665.6ms inference, 99.6ms postprocess per image at shape (1, 3, 480, 640)
Total Objects Detected: 5
Output image saved in the output folder.


In [3]:
# ==========================================================
# PART 4 - Frame-by-Frame Object Detection Pipeline
# ==========================================================

# Import required libraries
import cv2
import numpy as np
import time
from ultralytics import YOLO

# ----------------------------------------------------------
# Load YOLOv8 Model
# ----------------------------------------------------------

# Load the pre-trained YOLOv8 Nano model
model = YOLO("yolov8n.pt")

# ----------------------------------------------------------
# Load Input Video
# ----------------------------------------------------------

video = cv2.VideoCapture(r"C:\Users\ahmad\Downloads\188613-883402208.mp4")

# Check whether the video is loaded successfully
if not video.isOpened():
    print("Error: Unable to open video.")
    exit()

# ----------------------------------------------------------
# Video Properties
# ----------------------------------------------------------

frame_width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_input = int(video.get(cv2.CAP_PROP_FPS))

# Save processed video
output_video = cv2.VideoWriter(
    "output/output_video.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps_input,
    (frame_width, frame_height)
)

# ----------------------------------------------------------
# Start Timer
# ----------------------------------------------------------

previous_time = time.time()

# ==========================================================
# Process Every Frame
# ==========================================================

while True:

    # Read one frame
    success, frame = video.read()

    # Stop when video ends
    if not success:
        break

    # ------------------------------------------------------
    # OpenCV Image Processing
    # ------------------------------------------------------

    # Convert frame to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Apply Gaussian Blur
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # Adaptive Threshold
    threshold = cv2.adaptiveThreshold(
        blur,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        11,
        2
    )

    # Morphological Closing
    kernel = np.ones((3, 3), np.uint8)

    morph = cv2.morphologyEx(
        threshold,
        cv2.MORPH_CLOSE,
        kernel
    )

    # Detect contours
    contours, _ = cv2.findContours(
        morph,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # ------------------------------------------------------
    # YOLO Object Detection
    # ------------------------------------------------------

    results = model(frame)

    annotated_frame = results[0].plot()

    object_count = len(results[0].boxes)

    # ------------------------------------------------------
    # FPS Calculation
    # ------------------------------------------------------

    current_time = time.time()

    fps = 1 / (current_time - previous_time)

    previous_time = current_time

    # ------------------------------------------------------
    # Overlay Information
    # ------------------------------------------------------

    cv2.putText(
        annotated_frame,
        f"Objects: {object_count}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    cv2.putText(
        annotated_frame,
        f"Contours: {len(contours)}",
        (20, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 0, 0),
        2
    )

    cv2.putText(
        annotated_frame,
        f"FPS: {fps:.2f}",
        (20, 120),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 0, 255),
        2
    )

    # ------------------------------------------------------
    # Save Frame
    # ------------------------------------------------------

    output_video.write(annotated_frame)

    # Display Output
    cv2.imshow("Frame-by-Frame Object Detection", annotated_frame)

    # Press Q to Exit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# ==========================================================
# Release Resources
# ==========================================================

video.release()
output_video.release()

cv2.destroyAllWindows()

print("Video processing completed successfully.")
print("Processed video saved in output/output_video.mp4")


0: 384x640 24 cars, 5 trucks, 645.4ms
Speed: 51.0ms preprocess, 645.4ms inference, 76.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 25 cars, 5 trucks, 243.6ms
Speed: 6.1ms preprocess, 243.6ms inference, 4.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 24 cars, 4 trucks, 265.9ms
Speed: 8.0ms preprocess, 265.9ms inference, 3.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 28 cars, 3 trucks, 253.8ms
Speed: 5.7ms preprocess, 253.8ms inference, 4.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 28 cars, 1 bus, 3 trucks, 270.7ms
Speed: 7.5ms preprocess, 270.7ms inference, 5.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 30 cars, 4 trucks, 280.8ms
Speed: 8.2ms preprocess, 280.8ms inference, 4.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 32 cars, 4 trucks, 329.0ms
Speed: 7.7ms preprocess, 329.0ms inference, 4.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 31 cars, 4 trucks, 2